In [ ]:
# -*- coding: utf-8 -*-
"""06_production_preparation.ipynb

# **VeritasFinancial: Production Preparation for Banking Fraud Detection**
## **From Development to Production-Ready System**

### **Notebook Overview**
This comprehensive notebook transforms our fraud detection models from development artifacts into production-ready components. We'll cover:

1. **Model Serialization & Versioning** - Saving models with metadata
2. **Pipeline Persistence** - Saving preprocessing and feature engineering pipelines
3. **Model Performance Benchmarking** - Comprehensive evaluation across multiple metrics
4. **Threshold Optimization** - Finding optimal decision thresholds for business constraints
5. **Model Interpretability** - SHAP analysis and feature importance for compliance
6. **API Development** - Creating FastAPI endpoints for model serving
7. **Performance Testing** - Load testing and latency analysis
8. **Model Monitoring Setup** - Drift detection and performance tracking
9. **Containerization** - Docker setup for deployment
10. **CI/CD Pipeline Configuration** - Automated deployment workflows

**Author:** VeritasFinancial Data Science Team
**Version:** 1.0.0
**Last Updated:** 2024-01-15
"""

# =============================================================================
# SECTION 1: Environment Setup and Imports
# =============================================================================

"""
This section sets up the entire environment for production preparation.
We import all necessary libraries and configure logging, warnings, and
random seeds for reproducibility.

Key components:
- Data manipulation: pandas, numpy
- Machine learning: scikit-learn, xgboost, lightgbm
- Deep learning: torch, transformers
- Model interpretation: shap, lime, eli5
- Production tools: joblib, pickle, json, yaml
- Monitoring: prometheus_client, psutil
- API: fastapi, uvicorn
- Testing: locust, pytest
"""

# System and utility imports
import os
import sys
import json
import yaml
import pickle
import joblib
import logging
import warnings
import datetime
import hashlib
import base64
import time
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union, Any
from dataclasses import dataclass, field, asdict
from collections import defaultdict
from functools import wraps

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Data manipulation imports
import numpy as np
import pandas as pd
from pandas import DataFrame, Series

# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Statistical imports
from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency

# Machine Learning imports
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    PowerTransformer, QuantileTransformer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    SelectKBest, SelectFromModel, RFE,
    mutual_info_classif, f_classif
)
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import (
    RandomForestClassifier, IsolationForest,
    VotingClassifier, StackingClassifier
)
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, GridSearchCV,
    RandomizedSearchCV, train_test_split, learning_curve,
    validation_curve, cross_validate
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve,
    matthews_corrcoef, cohen_kappa_score, brier_score_loss,
    log_loss, balanced_accuracy_score
)

# Imbalanced learning imports
from imblearn.over_sampling import (
    SMOTE, ADASYN, RandomOverSampler, BorderlineSMOTE,
    SVMSMOTE, KMeansSMOTE
)
from imblearn.under_sampling import (
    RandomUnderSampler, NearMiss, TomekLinks,
    EditedNearestNeighbours, CondensedNearestNeighbour
)
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

# XGBoost and LightGBM
import xgboost as xgb
import lightgbm as lgb

# Deep Learning imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# Transformers imports
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

# Model interpretation imports
import shap
import lime
import lime.lime_tabular
import eli5
from eli5.sklearn import PermutationImportance

# Production and deployment imports
import joblib
import pickle
import cloudpickle
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.pytorch

# API imports
from fastapi import FastAPI, HTTPException, Depends, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field, validator
import uvicorn

# Monitoring imports
from prometheus_client import (
    Counter, Histogram, Gauge, generate_latest,
    CONTENT_TYPE_LATEST, start_http_server
)
import psutil

# Database imports
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine
import redis

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/production_preparation.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# =============================================================================
# SECTION 2: Configuration Management
# =============================================================================

"""
Configuration management is crucial for production systems. We'll create
a comprehensive configuration system that handles:

1. Model configurations (hyperparameters, architecture)
2. Data configurations (paths, schemas, validation rules)
3. Feature configurations (feature names, types, engineering steps)
4. Deployment configurations (environment, scaling, resources)
5. Monitoring configurations (metrics, alerts, thresholds)
6. Security configurations (authentication, encryption)

This configuration system ensures reproducibility and easy updates
without code changes.
"""

@dataclass
class ModelConfig:
    """
    Configuration class for model parameters and metadata.
    
    This dataclass centralizes all model-related configuration,
    making it easy to version and update models without code changes.
    
    Attributes:
        model_name: Name of the model for identification
        model_version: Semantic version of the model
        model_type: Type of model (xgboost, neural_network, ensemble)
        input_features: List of feature names used by the model
        target_column: Name of the target column
        hyperparameters: Dictionary of model hyperparameters
        feature_importance_method: Method for calculating feature importance
        threshold_optimization_metric: Metric for threshold optimization
        class_weight: Class weights for imbalanced data
        random_state: Random seed for reproducibility
        created_at: Timestamp of model creation
        updated_at: Timestamp of last update
        tags: Additional metadata tags
    """
    
    # Model identification
    model_name: str = "fraud_detection_model"
    model_version: str = "1.0.0"
    model_type: str = "xgboost"  # Options: xgboost, lightgbm, neural_network, ensemble
    
    # Feature configuration
    input_features: List[str] = field(default_factory=list)
    target_column: str = "is_fraud"
    categorical_features: List[str] = field(default_factory=list)
    numerical_features: List[str] = field(default_factory=list)
    text_features: List[str] = field(default_factory=list)
    temporal_features: List[str] = field(default_factory=list)
    
    # Model hyperparameters
    hyperparameters: Dict[str, Any] = field(default_factory=lambda: {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "scale_pos_weight": 1,
        "random_state": 42,
        "n_jobs": -1
    })
    
    # Training configuration
    validation_split: float = 0.2
    test_split: float = 0.1
    cross_validation_folds: int = 5
    early_stopping_rounds: int = 50
    
    # Feature engineering
    feature_selection_method: Optional[str] = None
    feature_selection_params: Dict[str, Any] = field(default_factory=dict)
    feature_importance_method: str = "shap"  # Options: shap, gain, weight, cover
    
    # Threshold optimization
    threshold_optimization_metric: str = "f1"  # Options: f1, precision, recall, business_cost
    business_cost_matrix: Dict[str, float] = field(default_factory=lambda: {
        "false_positive_cost": 1.0,  # Cost of investigating legitimate transaction
        "false_negative_cost": 100.0  # Cost of missing fraud
    })
    
    # Class imbalance
    class_weight: Optional[Union[str, Dict[int, float]]] = "balanced"
    sampling_strategy: Optional[str] = "smote"  # Options: smote, random_oversample, none
    
    # Reproducibility
    random_state: int = 42
    
    # Metadata
    created_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    updated_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    tags: Dict[str, str] = field(default_factory=dict)
    
    def __post_init__(self):
        """Validate configuration after initialization"""
        self._validate_config()
        self._set_feature_lists()
    
    def _validate_config(self):
        """Validate configuration parameters"""
        assert self.validation_split + self.test_split < 1.0, \
            "Validation and test splits must sum to less than 1.0"
        assert self.cross_validation_folds > 1, \
            "Cross validation folds must be greater than 1"
        assert self.random_state >= 0, \
            "Random state must be non-negative"
    
    def _set_feature_lists(self):
        """Set feature lists based on input features"""
        if not self.input_features:
            return
        
        # Auto-detect feature types if not specified
        if not self.categorical_features:
            self.categorical_features = []  # Will be set during data loading
        if not self.numerical_features:
            self.numerical_features = []  # Will be set during data loading
    
    def to_dict(self) -> Dict:
        """Convert config to dictionary for serialization"""
        return asdict(self)
    
    @classmethod
    def from_dict(cls, config_dict: Dict) -> 'ModelConfig':
        """Create config from dictionary"""
        return cls(**config_dict)
    
    def save(self, filepath: str):
        """Save configuration to file"""
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)
    
    @classmethod
    def load(cls, filepath: str) -> 'ModelConfig':
        """Load configuration from file"""
        with open(filepath, 'r') as f:
            config_dict = json.load(f)
        return cls.from_dict(config_dict)


@dataclass
class DataConfig:
    """
    Configuration for data loading and preprocessing.
    
    Attributes:
        data_path: Path to raw data
        processed_data_path: Path to processed data
        feature_store_path: Path to feature store
        database_connection: Database connection string
        batch_size: Batch size for data loading
        validation_rules: Dictionary of data validation rules
        schema: Dictionary defining data schema
    """
    
    data_path: str = "data/raw/transactions.csv"
    processed_data_path: str = "data/processed/"
    feature_store_path: str = "data/features/"
    database_connection: Optional[str] = None
    batch_size: int = 10000
    validation_rules: Dict[str, Any] = field(default_factory=lambda: {
        "amount": {"min": 0.01, "max": 1000000},
        "transaction_time": {"min_year": 2000, "max_year": 2025},
        "customer_id": {"regex": "^CUST\\d{8}$"}
    })
    schema: Dict[str, str] = field(default_factory=lambda: {
        "transaction_id": "string",
        "customer_id": "string",
        "amount": "float",
        "transaction_time": "datetime",
        "merchant_id": "string",
        "merchant_category": "string",
        "country": "string",
        "city": "string",
        "device_id": "string",
        "ip_address": "string",
        "is_fraud": "integer"
    })


@dataclass
class DeploymentConfig:
    """
    Configuration for model deployment.
    
    Attributes:
        environment: Deployment environment (dev, staging, production)
        api_host: API host address
        api_port: API port
        workers: Number of worker processes
        max_requests: Maximum requests per worker
        timeout: Request timeout in seconds
        model_cache_size: Number of models to cache
        feature_cache_ttl: Feature cache TTL in seconds
        enable_monitoring: Whether to enable monitoring
        enable_authentication: Whether to enable authentication
        enable_rate_limiting: Whether to enable rate limiting
        rate_limit: Maximum requests per minute
    """
    
    environment: str = "development"
    api_host: str = "0.0.0.0"
    api_port: int = 8000
    workers: int = 4
    max_requests: int = 1000
    timeout: int = 30
    model_cache_size: int = 3
    feature_cache_ttl: int = 3600
    enable_monitoring: bool = True
    enable_authentication: bool = False
    enable_rate_limiting: bool = True
    rate_limit: int = 1000


# =============================================================================
# SECTION 3: Production Model Wrapper
# =============================================================================

"""
The ProductionModelWrapper class encapsulates all model-related functionality
for production deployment. It handles:

1. Model loading and versioning
2. Preprocessing and feature engineering
3. Prediction with confidence scores
4. Model interpretation (SHAP, LIME)
5. Performance monitoring
6. Model serialization/deserialization
7. Error handling and logging

This wrapper ensures consistent behavior across all deployment environments
and provides a clean interface for API endpoints.
"""

class ProductionModelWrapper:
    """
    Production-ready model wrapper with comprehensive functionality.
    
    This class wraps machine learning models with all necessary production
    features including preprocessing, interpretation, monitoring, and
    error handling.
    
    Attributes:
        config: Model configuration
        model: Underlying machine learning model
        preprocessor: Preprocessing pipeline
        feature_engineer: Feature engineering pipeline
        label_encoder: Label encoder for target variable
        scaler: Feature scaler
        threshold: Decision threshold
        feature_importance: Feature importance scores
        shap_explainer: SHAP explainer for model interpretation
        metrics: Performance metrics
        created_at: Creation timestamp
        updated_at: Last update timestamp
        version: Model version
    """
    
    def __init__(
        self,
        config: ModelConfig,
        model: Optional[Any] = None,
        preprocessor: Optional[Pipeline] = None,
        feature_engineer: Optional[Pipeline] = None
    ):
        """
        Initialize the production model wrapper.
        
        Args:
            config: Model configuration object
            model: Pre-trained model (optional)
            preprocessor: Preprocessing pipeline (optional)
            feature_engineer: Feature engineering pipeline (optional)
        """
        self.config = config
        self.model = model
        self.preprocessor = preprocessor
        self.feature_engineer = feature_engineer
        self.label_encoder = None
        self.scaler = None
        self.threshold = 0.5
        self.feature_importance = {}
        self.shap_explainer = None
        self.metrics = {}
        self.created_at = datetime.datetime.now()
        self.updated_at = datetime.datetime.now()
        self.version = config.model_version
        
        # Monitoring metrics
        self.prediction_count = 0
        self.error_count = 0
        self.latency_history = []
        self.prediction_history = []
        
        # Initialize logger
        self.logger = logging.getLogger(f"{__name__}.{self.config.model_name}")
        
        self.logger.info(f"Initialized ProductionModelWrapper for {self.config.model_name} v{self.version}")
    
    def fit_preprocessor(self, X: pd.DataFrame, y: Optional[pd.Series] = None):
        """
        Fit preprocessing pipeline on training data.
        
        This method creates and fits a comprehensive preprocessing pipeline
        that handles missing values, outliers, feature scaling, and encoding.
        
        Args:
            X: Training features
            y: Training labels (optional)
        """
        self.logger.info("Fitting preprocessing pipeline...")
        
        # Create preprocessing steps based on feature types
        preprocessing_steps = []
        
        # Numerical features preprocessing
        if self.config.numerical_features:
            numerical_transformer = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', RobustScaler()),  # RobustScaler handles outliers better
                ('power', PowerTransformer(method='yeo-johnson'))  # Handle skewness
            ])
            preprocessing_steps.append(('num', numerical_transformer, self.config.numerical_features))
        
        # Categorical features preprocessing
        if self.config.categorical_features:
            categorical_transformer = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ])
            preprocessing_steps.append(('cat', categorical_transformer, self.config.categorical_features))
        
        # Create column transformer
        if preprocessing_steps:
            self.preprocessor = ColumnTransformer(
                transformers=preprocessing_steps,
                remainder='drop'  # Drop unspecified columns
            )
            
            # Fit the preprocessor
            self.preprocessor.fit(X)
            
            self.logger.info(f"Preprocessing pipeline fitted with {len(preprocessing_steps)} transformers")
        else:
            self.logger.warning("No preprocessing steps defined")
    
    def fit_feature_engineer(self, X: pd.DataFrame, y: Optional[pd.Series] = None):
        """
        Fit feature engineering pipeline.
        
        This method creates advanced features including:
        - Interaction features
        - Polynomial features
        - Aggregated features
        - Temporal features
        
        Args:
            X: Training features
            y: Training labels (optional)
        """
        self.logger.info("Fitting feature engineering pipeline...")
        
        # Create feature engineering pipeline
        feature_steps = []
        
        # Add interaction features for numerical columns
        if len(self.config.numerical_features) >= 2:
            feature_steps.append(('interactions', PolynomialFeatures(
                degree=2, 
                interaction_only=True, 
                include_bias=False
            )))
        
        # Add feature selection based on importance
        if self.config.feature_selection_method == 'mutual_info':
            selector = SelectKBest(
                score_func=mutual_info_classif,
                k=min(20, len(X.columns))
            )
            feature_steps.append(('selection', selector))
        elif self.config.feature_selection_method == 'model_based':
            base_model = RandomForestClassifier(n_estimators=100, random_state=42)
            selector = SelectFromModel(base_model, threshold='median')
            feature_steps.append(('selection', selector))
        
        # Create pipeline
        if feature_steps:
            self.feature_engineer = Pipeline(feature_steps)
            self.feature_engineer.fit(X, y)
            self.logger.info(f"Feature engineering pipeline fitted with {len(feature_steps)} steps")
        else:
            self.logger.info("No feature engineering steps configured")
    
    def train(self, X_train: pd.DataFrame, y_train: pd.Series, 
              X_val: Optional[pd.DataFrame] = None, 
              y_val: Optional[pd.Series] = None) -> Dict[str, float]:
        """
        Train the model with comprehensive training loop.
        
        This method handles:
        - Model initialization based on config
        - Training with early stopping
        - Validation monitoring
        - Metrics calculation
        - Model persistence
        
        Args:
            X_train: Training features
            y_train: Training labels
            X_val: Validation features (optional)
            y_val: Validation labels (optional)
            
        Returns:
            Dictionary of training metrics
        """
        self.logger.info(f"Starting model training for {self.config.model_name}...")
        
        # Preprocess data if preprocessor exists
        if self.preprocessor:
            X_train_processed = self.preprocessor.transform(X_train)
            if X_val is not None:
                X_val_processed = self.preprocessor.transform(X_val)
        else:
            X_train_processed = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
            if X_val is not None:
                X_val_processed = X_val.values if isinstance(X_val, pd.DataFrame) else X_val
        
        # Engineer features if feature engineer exists
        if self.feature_engineer:
            X_train_processed = self.feature_engineer.fit_transform(X_train_processed, y_train)
            if X_val is not None:
                X_val_processed = self.feature_engineer.transform(X_val_processed)
        
        # Train based on model type
        if self.config.model_type == 'xgboost':
            self.model, metrics = self._train_xgboost(
                X_train_processed, y_train,
                X_val_processed, y_val
            )
        elif self.config.model_type == 'lightgbm':
            self.model, metrics = self._train_lightgbm(
                X_train_processed, y_train,
                X_val_processed, y_val
            )
        elif self.config.model_type == 'neural_network':
            self.model, metrics = self._train_neural_network(
                X_train_processed, y_train,
                X_val_processed, y_val
            )
        elif self.config.model_type == 'ensemble':
            self.model, metrics = self._train_ensemble(
                X_train_processed, y_train,
                X_val_processed, y_val
            )
        else:
            raise ValueError(f"Unknown model type: {self.config.model_type}")
        
        # Calculate feature importance
        self._calculate_feature_importance(X_train_processed, y_train)
        
        # Optimize threshold if validation data provided
        if X_val is not None and y_val is not None:
            self._optimize_threshold(X_val_processed, y_val)
        
        # Store metrics
        self.metrics.update(metrics)
        self.updated_at = datetime.datetime.now()
        
        self.logger.info(f"Model training completed. Metrics: {metrics}")
        
        return metrics
    
    def _train_xgboost(self, X_train, y_train, X_val=None, y_val=None) -> Tuple[xgb.Booster, Dict]:
        """
        Train XGBoost model with advanced parameters.
        
        Args:
            X_train: Training features
            y_train: Training labels
            X_val: Validation features
            y_val: Validation labels
            
        Returns:
            Trained XGBoost model and metrics dictionary
        """
        self.logger.info("Training XGBoost model...")
        
        # Prepare parameters
        params = self.config.hyperparameters.copy()
        
        # Calculate scale_pos_weight for imbalanced data
        if params.get('scale_pos_weight') == 'auto':
            neg_count = np.sum(y_train == 0)
            pos_count = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg_count / pos_count if pos_count > 0 else 1
        
        # Create DMatrix
        dtrain = xgb.DMatrix(X_train, label=y_train)
        
        # Validation set
        evals = [(dtrain, 'train')]
        if X_val is not None and y_val is not None:
            dval = xgb.DMatrix(X_val, label=y_val)
            evals.append((dval, 'eval'))
        
        # Train model
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=params.get('n_estimators', 100),
            evals=evals,
            early_stopping_rounds=self.config.early_stopping_rounds,
            verbose_eval=False
        )
        
        # Calculate metrics
        metrics = self._calculate_model_metrics(model, X_train, y_train, 'train')
        if X_val is not None and y_val is not None:
            val_metrics = self._calculate_model_metrics(model, X_val, y_val, 'validation')
            metrics.update(val_metrics)
        
        return model, metrics
    
    def _calculate_model_metrics(self, model, X, y_true, prefix='') -> Dict:
        """
        Calculate comprehensive model metrics.
        
        Args:
            model: Trained model
            X: Features
            y_true: True labels
            prefix: Metric prefix (train/val/test)
            
        Returns:
            Dictionary of metrics
        """
        # Get predictions
        if isinstance(model, xgb.Booster):
            if not isinstance(X, xgb.DMatrix):
                X = xgb.DMatrix(X)
            y_pred_proba = model.predict(X)
        else:
            y_pred_proba = model.predict_proba(X)[:, 1]
        
        y_pred = (y_pred_proba >= self.threshold).astype(int)
        
        # Calculate all metrics
        metrics = {}
        prefix = f"{prefix}_" if prefix else ""
        
        metrics[f"{prefix}accuracy"] = accuracy_score(y_true, y_pred)
        metrics[f"{prefix}precision"] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f"{prefix}recall"] = recall_score(y_true, y_pred, zero_division=0)
        metrics[f"{prefix}f1"] = f1_score(y_true, y_pred, zero_division=0)
        metrics[f"{prefix}roc_auc"] = roc_auc_score(y_true, y_pred_proba)
        metrics[f"{prefix}pr_auc"] = average_precision_score(y_true, y_pred_proba)
        metrics[f"{prefix}mcc"] = matthews_corrcoef(y_true, y_pred)
        metrics[f"{prefix}cohen_kappa"] = cohen_kappa_score(y_true, y_pred)
        metrics[f"{prefix}brier"] = brier_score_loss(y_true, y_pred_proba)
        metrics[f"{prefix}log_loss"] = log_loss(y_true, y_pred_proba)
        metrics[f"{prefix}balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)
        
        return metrics
    
    def _calculate_feature_importance(self, X: np.ndarray, y: np.ndarray):
        """
        Calculate feature importance using multiple methods.
        
        This method provides comprehensive feature importance analysis
        using:
        1. Model-based importance (gain, weight, cover for tree models)
        2. SHAP values (most accurate but computationally expensive)
        3. Permutation importance (model-agnostic)
        4. Mutual information (information theory based)
        
        Args:
            X: Feature matrix
            y: Target labels
        """
        self.logger.info("Calculating feature importance...")
        
        importance_methods = {}
        
        # 1. Model-based importance (if applicable)
        if hasattr(self.model, 'get_score'):
            # XGBoost importance
            importance_methods['gain'] = self.model.get_score(importance_type='gain')
            importance_methods['weight'] = self.model.get_score(importance_type='weight')
            importance_methods['cover'] = self.model.get_score(importance_type='cover')
        elif hasattr(self.model, 'feature_importances_'):
            # sklearn-like models
            importance_methods['model'] = dict(zip(
                self.config.input_features,
                self.model.feature_importances_
            ))
        
        # 2. SHAP values (if configured)
        if self.config.feature_importance_method == 'shap':
            try:
                # Create SHAP explainer based on model type
                if isinstance(self.model, xgb.Booster):
                    self.shap_explainer = shap.TreeExplainer(self.model)
                elif isinstance(self.model, (RandomForestClassifier, lgb.LGBMClassifier)):
                    self.shap_explainer = shap.TreeExplainer(self.model)
                else:
                    # Use KernelExplainer as fallback
                    self.shap_explainer = shap.KernelExplainer(
                        self.model.predict_proba, 
                        X[:100]  # Use subset for kernel explainer
                    )
                
                # Calculate SHAP values on sample
                X_sample = X[:100]  # Use sample for efficiency
                shap_values = self.shap_explainer.shap_values(X_sample)
                
                # Average absolute SHAP values
                if isinstance(shap_values, list):
                    # For multi-class, take values for positive class
                    shap_importance = np.abs(shap_values[1]).mean(axis=0)
                else:
                    shap_importance = np.abs(shap_values).mean(axis=0)
                
                importance_methods['shap'] = dict(zip(
                    self.config.input_features,
                    shap_importance
                ))
            except Exception as e:
                self.logger.warning(f"SHAP calculation failed: {e}")
        
        # 3. Permutation importance
        try:
            perm_importance = PermutationImportance(
                self.model, 
                random_state=self.config.random_state
            ).fit(X, y)
            
            importance_methods['permutation'] = dict(zip(
                self.config.input_features,
                perm_importance.feature_importances_
            ))
        except Exception as e:
            self.logger.warning(f"Permutation importance calculation failed: {e}")
        
        # Store all importance methods
        self.feature_importance = importance_methods
        
        # Log top features
        if importance_methods:
            primary_method = list(importance_methods.keys())[0]
            top_features = sorted(
                importance_methods[primary_method].items(),
                key=lambda x: x[1],
                reverse=True
            )[:10]
            
            self.logger.info(f"Top 10 features ({primary_method}):")
            for feature, importance in top_features:
                self.logger.info(f"  {feature}: {importance:.4f}")
    
    def _optimize_threshold(self, X_val: np.ndarray, y_val: np.ndarray):
        """
        Optimize decision threshold based on business metrics.
        
        This method finds the optimal threshold that maximizes the
        specified metric (f1, precision, recall) or minimizes
        business cost based on the cost matrix.
        
        Args:
            X_val: Validation features
            y_val: Validation labels
        """
        self.logger.info("Optimizing decision threshold...")
        
        # Get prediction probabilities
        y_pred_proba = self.predict_proba(X_val)
        
        # Try thresholds from 0.1 to 0.9
        thresholds = np.linspace(0.1, 0.9, 50)
        
        if self.config.threshold_optimization_metric == 'business_cost':
            # Optimize for business cost
            costs = []
            for threshold in thresholds:
                y_pred = (y_pred_proba >= threshold).astype(int)
                
                # Calculate confusion matrix
                tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
                
                # Calculate total cost
                total_cost = (
                    fp * self.config.business_cost_matrix['false_positive_cost'] +
                    fn * self.config.business_cost_matrix['false_negative_cost']
                )
                costs.append(total_cost)
            
            # Find threshold with minimum cost
            optimal_idx = np.argmin(costs)
            self.threshold = thresholds[optimal_idx]
            
            self.logger.info(f"Optimal threshold (min cost): {self.threshold:.4f}")
            
        else:
            # Optimize for F1, precision, or recall
            scores = []
            for threshold in thresholds:
                y_pred = (y_pred_proba >= threshold).astype(int)
                
                if self.config.threshold_optimization_metric == 'f1':
                    score = f1_score(y_val, y_pred, zero_division=0)
                elif self.config.threshold_optimization_metric == 'precision':
                    score = precision_score(y_val, y_pred, zero_division=0)
                elif self.config.threshold_optimization_metric == 'recall':
                    score = recall_score(y_val, y_pred, zero_division=0)
                else:
                    score = f1_score(y_val, y_pred, zero_division=0)
                
                scores.append(score)
            
            # Find threshold with maximum score
            optimal_idx = np.argmax(scores)
            self.threshold = thresholds[optimal_idx]
            
            self.logger.info(f"Optimal threshold (max {self.config.threshold_optimization_metric}): "
                           f"{self.threshold:.4f} (score: {scores[optimal_idx]:.4f})")
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """
        Make binary predictions using optimized threshold.
        
        Args:
            X: Input features
            
        Returns:
            Binary predictions (0/1)
        """
        # Get prediction probabilities
        probabilities = self.predict_proba(X)
        
        # Apply threshold
        predictions = (probabilities >= self.threshold).astype(int)
        
        # Update monitoring metrics
        self.prediction_count += 1
        self.prediction_history.append({
            'timestamp': datetime.datetime.now().isoformat(),
            'predictions': predictions.tolist(),
            'probabilities': probabilities.tolist()
        })
        
        return predictions
    
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predict fraud probabilities with preprocessing.
        
        This method handles the complete prediction pipeline:
        1. Input validation
        2. Preprocessing
        3. Feature engineering
        4. Model prediction
        5. Post-processing
        
        Args:
            X: Input features DataFrame
            
        Returns:
            Array of fraud probabilities
        """
        start_time = time.time()
        
        try:
            # Input validation
            if not isinstance(X, pd.DataFrame):
                X = pd.DataFrame(X)
            
            # Ensure all required features are present
            missing_features = set(self.config.input_features) - set(X.columns)
            if missing_features:
                raise ValueError(f"Missing required features: {missing_features}")
            
            # Select only required features in correct order
            X = X[self.config.input_features]
            
            # Handle missing values
            X = self._handle_missing_values(X)
            
            # Apply preprocessing
            if self.preprocessor:
                X_processed = self.preprocessor.transform(X)
            else:
                X_processed = X.values
            
            # Apply feature engineering
            if self.feature_engineer:
                X_processed = self.feature_engineer.transform(X_processed)
            
            # Make prediction based on model type
            if isinstance(self.model, xgb.Booster):
                # XGBoost requires DMatrix
                dmatrix = xgb.DMatrix(X_processed)
                probabilities = self.model.predict(dmatrix)
            elif hasattr(self.model, 'predict_proba'):
                # sklearn-like models
                probabilities = self.model.predict_proba(X_processed)[:, 1]
            else:
                # Fallback to predict and convert
                predictions = self.model.predict(X_processed)
                probabilities = predictions  # Assume already probabilities
            
            # Ensure probabilities are in [0, 1]
            probabilities = np.clip(probabilities, 0, 1)
            
            # Update latency history
            latency = time.time() - start_time
            self.latency_history.append(latency)
            
            # Keep only last 1000 latencies
            if len(self.latency_history) > 1000:
                self.latency_history.pop(0)
            
            return probabilities
            
        except Exception as e:
            self.error_count += 1
            self.logger.error(f"Prediction failed: {str(e)}")
            raise
    
    def _handle_missing_values(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Handle missing values in input data.
        
        Args:
            X: Input DataFrame
            
        Returns:
            DataFrame with handled missing values
        """
        # Check for missing values
        missing_cols = X.columns[X.isnull().any()].tolist()
        
        if missing_cols:
            self.logger.warning(f"Missing values detected in columns: {missing_cols}")
            
            # Fill numerical features with median
            for col in missing_cols:
                if col in self.config.numerical_features:
                    median_val = X[col].median()
                    X[col].fillna(median_val, inplace=True)
                elif col in self.config.categorical_features:
                    X[col].fillna('missing', inplace=True)
        
        return X
    
    def explain_prediction(self, X: pd.DataFrame, method: str = 'shap') -> Dict:
        """
        Generate explanation for a single prediction.
        
        This method provides model interpretability using:
        - SHAP values (most accurate)
        - LIME (local explanations)
        - Feature contributions (model-specific)
        
        Args:
            X: Input features for single instance
            method: Explanation method ('shap', 'lime', 'contributions')
            
        Returns:
            Dictionary with explanation details
        """
        if method == 'shap':
            return self._explain_with_shap(X)
        elif method == 'lime':
            return self._explain_with_lime(X)
        elif method == 'contributions':
            return self._explain_with_contributions(X)
        else:
            raise ValueError(f"Unknown explanation method: {method}")
    
    def _explain_with_shap(self, X: pd.DataFrame) -> Dict:
        """
        Generate SHAP-based explanation.
        
        Args:
            X: Input features
            
        Returns:
            SHAP explanation dictionary
        """
        if self.shap_explainer is None:
            raise ValueError("SHAP explainer not initialized. Run with shap importance first.")
        
        # Prepare data
        if self.preprocessor:
            X_processed = self.preprocessor.transform(X)
        else:
            X_processed = X.values
        
        # Calculate SHAP values
        shap_values = self.shap_explainer.shap_values(X_processed)
        
        # Handle multi-class output
        if isinstance(shap_values, list):
            shap_values = shap_values[1]  # Take values for positive class
        
        # Create explanation
        explanation = {
            'method': 'shap',
            'base_value': float(self.shap_explainer.expected_value 
                               if not isinstance(self.shap_explainer.expected_value, list)
                               else self.shap_explainer.expected_value[1]),
            'shap_values': shap_values[0].tolist(),
            'features': X.columns.tolist(),
            'feature_values': X.values[0].tolist(),
            'prediction': float(self.predict_proba(X)[0])
        }
        
        # Add top contributing features
        feature_contributions = [
            {
                'feature': feature,
                'value': float(value),
                'shap_value': float(shap_val),
                'contribution': 'positive' if shap_val > 0 else 'negative'
            }
            for feature, value, shap_val in zip(
                X.columns, 
                X.values[0], 
                shap_values[0]
            )
        ]
        
        feature_contributions.sort(key=lambda x: abs(x['shap_value']), reverse=True)
        explanation['top_contributions'] = feature_contributions[:10]
        
        return explanation
    
    def get_performance_metrics(self) -> Dict:
        """
        Get real-time performance metrics.
        
        Returns:
            Dictionary of performance metrics
        """
        metrics = {
            'total_predictions': self.prediction_count,
            'error_count': self.error_count,
            'error_rate': self.error_count / max(self.prediction_count, 1),
            'average_latency_ms': np.mean(self.latency_history) * 1000 if self.latency_history else 0,
            'p95_latency_ms': np.percentile(self.latency_history, 95) * 1000 if self.latency_history else 0,
            'p99_latency_ms': np.percentile(self.latency_history, 99) * 1000 if self.latency_history else 0,
            'max_latency_ms': max(self.latency_history) * 1000 if self.latency_history else 0,
            'min_latency_ms': min(self.latency_history) * 1000 if self.latency_history else 0,
            'threshold': self.threshold,
            'model_version': self.version,
            'last_updated': self.updated_at.isoformat()
        }
        
        return metrics
    
    def save(self, filepath: str):
        """
        Save complete model pipeline to disk.
        
        This method saves:
        - Model configuration
        - Trained model
        - Preprocessing pipeline
        - Feature engineering pipeline
        - Feature importance
        - Threshold and metadata
        
        Args:
            filepath: Path to save the model
        """
        self.logger.info(f"Saving model to {filepath}")
        
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        
        # Prepare save data
        save_data = {
            'config': self.config.to_dict(),
            'model': self.model,
            'preprocessor': self.preprocessor,
            'feature_engineer': self.feature_engineer,
            'label_encoder': self.label_encoder,
            'scaler': self.scaler,
            'threshold': self.threshold,
            'feature_importance': self.feature_importance,
            'metrics': self.metrics,
            'version': self.version,
            'created_at': self.created_at.isoformat(),
            'updated_at': self.updated_at.isoformat()
        }
        
        # Save using cloudpickle for better compatibility
        with open(filepath, 'wb') as f:
            cloudpickle.dump(save_data, f)
        
        self.logger.info(f"Model saved successfully to {filepath}")
    
    @classmethod
    def load(cls, filepath: str) -> 'ProductionModelWrapper':
        """
        Load complete model pipeline from disk.
        
        Args:
            filepath: Path to load the model from
            
        Returns:
            Loaded ProductionModelWrapper instance
        """
        logger.info(f"Loading model from {filepath}")
        
        # Load save data
        with open(filepath, 'rb') as f:
            save_data = cloudpickle.load(f)
        
        # Create instance with loaded config
        instance = cls(
            config=ModelConfig.from_dict(save_data['config']),
            model=save_data['model'],
            preprocessor=save_data['preprocessor'],
            feature_engineer=save_data['feature_engineer']
        )
        
        # Load remaining attributes
        instance.label_encoder = save_data.get('label_encoder')
        instance.scaler = save_data.get('scaler')
        instance.threshold = save_data.get('threshold', 0.5)
        instance.feature_importance = save_data.get('feature_importance', {})
        instance.metrics = save_data.get('metrics', {})
        instance.version = save_data.get('version', '1.0.0')
        instance.created_at = datetime.datetime.fromisoformat(save_data.get('created_at', datetime.datetime.now().isoformat()))
        instance.updated_at = datetime.datetime.fromisoformat(save_data.get('updated_at', datetime.datetime.now().isoformat()))
        
        logger.info(f"Model loaded successfully from {filepath}")
        
        return instance


# =============================================================================
# SECTION 4: Model Versioning and Registry
# =============================================================================

"""
Model versioning is critical for production deployments. This section implements
a model registry that tracks all model versions, their performance metrics,
and deployment status. It enables:

1. Version control for models
2. Staging to production promotion
3. A/B testing support
4. Model rollback capabilities
5. Performance tracking over time
"""

class ModelRegistry:
    """
    Model registry for tracking model versions and metadata.
    
    This class provides a centralized registry for all model versions,
    enabling version control, staging/production promotion, and
    performance tracking.
    
    Attributes:
        registry_path: Path to registry storage
        models: Dictionary of registered models
        current_production: Current production model version
        current_staging: Current staging model version
    """
    
    def __init__(self, registry_path: str = "models/registry"):
        """
        Initialize model registry.
        
        Args:
            registry_path: Path to store registry data
        """
        self.registry_path = Path(registry_path)
        self.registry_path.mkdir(parents=True, exist_ok=True)
        
        self.models_file = self.registry_path / "models.json"
        self.models = self._load_registry()
        self.current_production = None
        self.current_staging = None
        
        self.logger = logging.getLogger(f"{__name__}.ModelRegistry")
        self.logger.info(f"Initialized model registry at {registry_path}")
    
    def _load_registry(self) -> Dict:
        """
        Load registry from disk.
        
        Returns:
            Dictionary of registered models
        """
        if self.models_file.exists():
            with open(self.models_file, 'r') as f:
                return json.load(f)
        return {}
    
    def _save_registry(self):
        """Save registry to disk."""
        with open(self.models_file, 'w') as f:
            json.dump(self.models, f, indent=2)
    
    def register_model(
        self,
        model_path: str,
        model_name: str,
        version: str,
        metrics: Dict,
        tags: Dict = None
    ) -> str:
        """
        Register a new model version.
        
        Args:
            model_path: Path to saved model file
            model_name: Name of the model
            version: Model version
            metrics: Performance metrics
            tags: Additional metadata tags
            
        Returns:
            Model ID
        """
        # Generate unique model ID
        model_id = hashlib.md5(
            f"{model_name}_{version}_{datetime.datetime.now().isoformat()}".encode()
        ).hexdigest()[:8]
        
        # Create model entry
        model_entry = {
            'model_id': model_id,
            'model_name': model_name,
            'version': version,
            'path': str(model_path),
            'metrics': metrics,
            'tags': tags or {},
            'stage': 'development',
            'created_at': datetime.datetime.now().isoformat(),
            'updated_at': datetime.datetime.now().isoformat(),
            'deployments': []
        }
        
        # Store in registry
        if model_name not in self.models:
            self.models[model_name] = {}
        
        self.models[model_name][version] = model_entry
        self._save_registry()
        
        self.logger.info(f"Registered model {model_name} v{version} with ID {model_id}")
        
        return model_id
    
    def promote_to_staging(self, model_name: str, version: str):
        """
        Promote a model version to staging.
        
        Args:
            model_name: Name of the model
            version: Model version to promote
        """
        if model_name not in self.models or version not in self.models[model_name]:
            raise ValueError(f"Model {model_name} v{version} not found")
        
        # Update model stage
        self.models[model_name][version]['stage'] = 'staging'
        self.models[model_name][version]['updated_at'] = datetime.datetime.now().isoformat()
        
        # Update current staging
        self.current_staging = (model_name, version)
        
        # Add deployment record
        self.models[model_name][version]['deployments'].append({
            'stage': 'staging',
            'timestamp': datetime.datetime.now().isoformat()
        })
        
        self._save_registry()
        self.logger.info(f"Promoted {model_name} v{version} to staging")
    
    def promote_to_production(self, model_name: str, version: str):
        """
        Promote a model version to production.
        
        Args:
            model_name: Name of the model
            version: Model version to promote
        """
        if model_name not in self.models or version not in self.models[model_name]:
            raise ValueError(f"Model {model_name} v{version} not found")
        
        # Demote current production if exists
        if self.current_production:
            old_name, old_version = self.current_production
            self.models[old_name][old_version]['stage'] = 'archived'
        
        # Promote new model
        self.models[model_name][version]['stage'] = 'production'
        self.models[model_name][version]['updated_at'] = datetime.datetime.now().isoformat()
        
        # Update current production
        self.current_production = (model_name, version)
        
        # Add deployment record
        self.models[model_name][version]['deployments'].append({
            'stage': 'production',
            'timestamp': datetime.datetime.now().isoformat()
        })
        
        self._save_registry()
        self.logger.info(f"Promoted {model_name} v{version} to production")
    
    def get_model(self, model_name: str, version: Optional[str] = None, 
                  stage: Optional[str] = None) -> Optional[Dict]:
        """
        Get model information.
        
        Args:
            model_name: Name of the model
            version: Specific version (optional)
            stage: Stage (production/staging/development) (optional)
            
        Returns:
            Model information dictionary
        """
        if model_name not in self.models:
            return None
        
        if version:
            return self.models[model_name].get(version)
        
        if stage:
            # Find model by stage
            for ver, info in self.models[model_name].items():
                if info.get('stage') == stage:
                    return info
            return None
        
        # Return latest version
        versions = sorted(self.models[model_name].keys(), reverse=True)
        if versions:
            return self.models[model_name][versions[0]]
        
        return None
    
    def list_models(self, model_name: Optional[str] = None) -> Dict:
        """
        List all models or versions of a specific model.
        
        Args:
            model_name: Optional model name to filter
            
        Returns:
            Dictionary of models
        """
        if model_name:
            return {model_name: self.models.get(model_name, {})}
        return self.models
    
    def compare_versions(self, model_name: str, versions: List[str]) -> pd.DataFrame:
        """
        Compare multiple versions of a model.
        
        Args:
            model_name: Name of the model
            versions: List of versions to compare
            
        Returns:
            DataFrame with comparison results
        """
        comparison = []
        
        for version in versions:
            if version in self.models.get(model_name, {}):
                model_info = self.models[model_name][version]
                metrics = model_info.get('metrics', {})
                
                comparison.append({
                    'version': version,
                    'stage': model_info.get('stage', 'unknown'),
                    'created_at': model_info.get('created_at'),
                    **metrics
                })
        
        return pd.DataFrame(comparison)
    
    def get_performance_history(self, model_name: str) -> pd.DataFrame:
        """
        Get performance history for a model.
        
        Args:
            model_name: Name of the model
            
        Returns:
            DataFrame with performance history
        """
        history = []
        
        if model_name in self.models:
            for version, info in self.models[model_name].items():
                history.append({
                    'version': version,
                    'stage': info.get('stage'),
                    'created_at': info.get('created_at'),
                    'deployments': len(info.get('deployments', [])),
                    **info.get('metrics', {})
                })
        
        return pd.DataFrame(history).sort_values('created_at')


# =============================================================================
# SECTION 5: FastAPI Application for Model Serving
# =============================================================================

"""
This section implements a production-ready FastAPI application for serving
fraud detection models. The API provides:

1. Health checks for monitoring
2. Real-time prediction endpoints
3. Batch prediction endpoints
4. Model explanation endpoints
5. Performance metrics endpoints
6. Model management endpoints
7. Authentication and rate limiting
8. Prometheus metrics for monitoring
"""

# Define Pydantic models for request/response validation
class TransactionFeatures(BaseModel):
    """
    Pydantic model for transaction features with validation.
    
    This model defines the expected input format for predictions
    and includes validation rules for each field.
    """
    
    transaction_id: str = Field(..., description="Unique transaction identifier")
    customer_id: str = Field(..., description="Customer identifier")
    amount: float = Field(..., gt=0, le=1000000, description="Transaction amount")
    currency: str = Field(..., min_length=3, max_length=3, description="Currency code")
    transaction_time: str = Field(..., description="Transaction timestamp")
    merchant_id: str = Field(..., description="Merchant identifier")
    merchant_category: str = Field(..., description="Merchant category")
    country: str = Field(..., min_length=2, max_length=2, description="Country code")
    city: str = Field(..., description="City name")
    device_id: str = Field(..., description="Device identifier")
    device_type: str = Field(..., description="Device type")
    ip_address: str = Field(..., description="IP address")
    
    @validator('amount')
    def validate_amount(cls, v):
        """Validate transaction amount"""
        if v <= 0:
            raise ValueError('Amount must be positive')
        if v > 1000000:
            raise ValueError('Amount exceeds maximum limit')
        return v
    
    @validator('currency')
    def validate_currency(cls, v):
        """Validate currency code"""
        valid_currencies = {'USD', 'EUR', 'GBP', 'JPY', 'CAD', 'AUD', 'CHF', 'CNY'}
        if v not in valid_currencies:
            raise ValueError(f'Invalid currency. Must be one of: {valid_currencies}')
        return v
    
    @validator('transaction_time')
    def validate_transaction_time(cls, v):
        """Validate transaction timestamp"""
        try:
            datetime.datetime.fromisoformat(v.replace('Z', '+00:00'))
        except:
            raise ValueError('Invalid timestamp format. Use ISO format')
        return v
    
    @validator('country')
    def validate_country(cls, v):
        """Validate country code"""
        if not v.isalpha() or len(v) != 2:
            raise ValueError('Country must be 2-letter ISO code')
        return v.upper()
    
    @validator('ip_address')
    def validate_ip(cls, v):
        """Validate IP address"""
        import ipaddress
        try:
            ipaddress.ip_address(v)
        except:
            raise ValueError('Invalid IP address')
        return v


class PredictionRequest(BaseModel):
    """
    Request model for single prediction.
    """
    
    transaction: TransactionFeatures
    explain: bool = Field(False, description="Whether to include explanation")
    explanation_method: str = Field("shap", description="Explanation method (shap/lime)")


class BatchPredictionRequest(BaseModel):
    """
    Request model for batch prediction.
    """
    
    transactions: List[TransactionFeatures]
    batch_size: int = Field(100, ge=1, le=1000, description="Batch size for processing")


class PredictionResponse(BaseModel):
    """
    Response model for predictions.
    """
    
    transaction_id: str
    fraud_probability: float
    prediction: int
    risk_level: str
    processing_time_ms: float
    model_version: str
    threshold: float
    explanation: Optional[Dict] = None


class BatchPredictionResponse(BaseModel):
    """
    Response model for batch predictions.
    """
    
    predictions: List[PredictionResponse]
    total_processing_time_ms: float
    average_processing_time_ms: float
    batch_size: int


# Prometheus metrics for monitoring
prediction_counter = Counter(
    'fraud_predictions_total',
    'Total number of fraud predictions',
    ['model_version', 'prediction']
)

prediction_latency = Histogram(
    'fraud_prediction_latency_seconds',
    'Prediction latency in seconds',
    ['model_version'],
    buckets=(0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1, 2.5, 5, 10)
)

model_info = Gauge(
    'fraud_model_info',
    'Model information',
    ['model_version', 'threshold']
)

error_counter = Counter(
    'fraud_errors_total',
    'Total number of errors',
    ['error_type']
)


# Create FastAPI application
def create_app(
    model_path: str,
    registry_path: str = "models/registry",
    config: DeploymentConfig = None
) -> FastAPI:
    """
    Create and configure FastAPI application.
    
    Args:
        model_path: Path to production model
        registry_path: Path to model registry
        config: Deployment configuration
        
    Returns:
        Configured FastAPI application
    """
    
    app = FastAPI(
        title="VeritasFinancial Fraud Detection API",
        description="Production-ready API for banking fraud detection",
        version="1.0.0",
        docs_url="/api/docs",
        redoc_url="/api/redoc"
    )
    
    # Load configuration
    if config is None:
        config = DeploymentConfig()
    
    # Add CORS middleware
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"] if config.environment == "development" else [],
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )
    
    # Load model and registry
    model_wrapper = ProductionModelWrapper.load(model_path)
    model_registry = ModelRegistry(registry_path)
    
    # Update model info metric
    model_info.labels(
        model_version=model_wrapper.version,
        threshold=model_wrapper.threshold
    ).set(1)
    
    # =====================================================================
    # API Endpoints
    # =====================================================================
    
    @app.get("/health", tags=["Monitoring"])
    async def health_check():
        """
        Health check endpoint for monitoring.
        
        Returns:
            Health status and system information
        """
        return {
            "status": "healthy",
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": model_wrapper.version,
            "environment": config.environment
        }
    
    @app.get("/ready", tags=["Monitoring"])
    async def readiness_check():
        """
        Readiness probe for Kubernetes.
        
        Returns:
            Readiness status
        """
        # Check if model is loaded
        if model_wrapper.model is None:
            raise HTTPException(status_code=503, detail="Model not loaded")
        
        return {"status": "ready"}
    
    @app.get("/metrics", tags=["Monitoring"])
    async def get_metrics():
        """
        Prometheus metrics endpoint.
        
        Returns:
            Prometheus metrics in text format
        """
        return StreamingResponse(
            generate_latest(),
            media_type=CONTENT_TYPE_LATEST
        )
    
    @app.post("/api/v1/predict", response_model=PredictionResponse, tags=["Prediction"])
    async def predict(request: PredictionRequest):
        """
        Make a single fraud prediction.
        
        Args:
            request: Prediction request with transaction features
            
        Returns:
            Prediction response with fraud probability and explanation
        """
        start_time = time.time()
        
        try:
            # Convert request to DataFrame
            features_df = pd.DataFrame([request.transaction.dict()])
            
            # Make prediction
            with prediction_latency.labels(model_version=model_wrapper.version).time():
                probability = model_wrapper.predict_proba(features_df)[0]
                prediction = (probability >= model_wrapper.threshold).astype(int)
            
            # Update metrics
            prediction_counter.labels(
                model_version=model_wrapper.version,
                prediction=str(prediction)
            ).inc()
            
            # Get explanation if requested
            explanation = None
            if request.explain:
                explanation = model_wrapper.explain_prediction(
                    features_df,
                    method=request.explanation_method
                )
            
            # Determine risk level
            if probability < 0.3:
                risk_level = "LOW"
            elif probability < 0.7:
                risk_level = "MEDIUM"
            else:
                risk_level = "HIGH"
            
            # Calculate processing time
            processing_time = (time.time() - start_time) * 1000  # Convert to ms
            
            return PredictionResponse(
                transaction_id=request.transaction.transaction_id,
                fraud_probability=float(probability),
                prediction=int(prediction),
                risk_level=risk_level,
                processing_time_ms=processing_time,
                model_version=model_wrapper.version,
                threshold=model_wrapper.threshold,
                explanation=explanation
            )
            
        except Exception as e:
            error_counter.labels(error_type=type(e).__name__).inc()
            logger.error(f"Prediction failed: {str(e)}")
            raise HTTPException(status_code=500, detail=str(e))
    
    @app.post("/api/v1/predict/batch", response_model=BatchPredictionResponse, tags=["Prediction"])
    async def predict_batch(request: BatchPredictionRequest):
        """
        Make batch predictions for multiple transactions.
        
        Args:
            request: Batch prediction request
            
        Returns:
            Batch prediction response with all predictions
        """
        start_time = time.time()
        
        try:
            # Convert all transactions to DataFrame
            transactions_dict = [t.dict() for t in request.transactions]
            features_df = pd.DataFrame(transactions_dict)
            
            # Process in batches to avoid memory issues
            predictions = []
            batch_size = min(request.batch_size, len(features_df))
            
            for i in range(0, len(features_df), batch_size):
                batch_df = features_df.iloc[i:i+batch_size]
                
                # Make predictions
                probabilities = model_wrapper.predict_proba(batch_df)
                batch_predictions = (probabilities >= model_wrapper.threshold).astype(int)
                
                # Create response objects
                for j, (idx, row) in enumerate(batch_df.iterrows()):
                    probability = probabilities[j]
                    
                    # Determine risk level
                    if probability < 0.3:
                        risk_level = "LOW"
                    elif probability < 0.7:
                        risk_level = "MEDIUM"
                    else:
                        risk_level = "HIGH"
                    
                    predictions.append(PredictionResponse(
                        transaction_id=row['transaction_id'],
                        fraud_probability=float(probability),
                        prediction=int(batch_predictions[j]),
                        risk_level=risk_level,
                        processing_time_ms=0,  # Will calculate total
                        model_version=model_wrapper.version,
                        threshold=model_wrapper.threshold,
                        explanation=None
                    ))
            
            # Update metrics
            for pred in predictions:
                prediction_counter.labels(
                    model_version=model_wrapper.version,
                    prediction=str(pred.prediction)
                ).inc()
            
            total_time = (time.time() - start_time) * 1000
            
            return BatchPredictionResponse(
                predictions=predictions,
                total_processing_time_ms=total_time,
                average_processing_time_ms=total_time / len(predictions),
                batch_size=len(predictions)
            )
            
        except Exception as e:
            error_counter.labels(error_type=type(e).__name__).inc()
            logger.error(f"Batch prediction failed: {str(e)}")
            raise HTTPException(status_code=500, detail=str(e))
    
    @app.get("/api/v1/model/info", tags=["Model Management"])
    async def get_model_info():
        """
        Get information about the current model.
        
        Returns:
            Model information including version, metrics, and feature importance
        """
        return {
            "model_name": model_wrapper.config.model_name,
            "model_version": model_wrapper.version,
            "model_type": model_wrapper.config.model_type,
            "threshold": model_wrapper.threshold,
            "created_at": model_wrapper.created_at.isoformat(),
            "updated_at": model_wrapper.updated_at.isoformat(),
            "metrics": model_wrapper.metrics,
            "feature_importance": model_wrapper.feature_importance,
            "input_features": model_wrapper.config.input_features,
            "num_features": len(model_wrapper.config.input_features)
        }
    
    @app.get("/api/v1/model/performance", tags=["Model Management"])
    async def get_model_performance():
        """
        Get real-time performance metrics.
        
        Returns:
            Performance metrics including latency and error rates
        """
        return model_wrapper.get_performance_metrics()
    
    @app.post("/api/v1/model/threshold", tags=["Model Management"])
    async def update_threshold(new_threshold: float):
        """
        Update the decision threshold.
        
        Args:
            new_threshold: New threshold value (0-1)
        """
        if not 0 <= new_threshold <= 1:
            raise HTTPException(status_code=400, detail="Threshold must be between 0 and 1")
        
        model_wrapper.threshold = new_threshold
        model_info.labels(
            model_version=model_wrapper.version,
            threshold=new_threshold
        ).set(1)
        
        return {"threshold": new_threshold, "updated_at": datetime.datetime.now().isoformat()}
    
    @app.get("/api/v1/models", tags=["Model Registry"])
    async def list_models():
        """
        List all registered models.
        
        Returns:
            List of all models in registry
        """
        return model_registry.list_models()
    
    @app.get("/api/v1/models/{model_name}/versions", tags=["Model Registry"])
    async def get_model_versions(model_name: str):
        """
        Get all versions of a specific model.
        
        Args:
            model_name: Name of the model
            
        Returns:
            List of model versions
        """
        return model_registry.list_models(model_name)
    
    @app.post("/api/v1/models/{model_name}/{version}/promote", tags=["Model Registry"])
    async def promote_model(model_name: str, version: str, stage: str):
        """
        Promote a model to a specific stage.
        
        Args:
            model_name: Name of the model
            version: Model version
            stage: Target stage (staging/production)
        """
        if stage not in ['staging', 'production']:
            raise HTTPException(status_code=400, detail="Stage must be 'staging' or 'production'")
        
        if stage == 'staging':
            model_registry.promote_to_staging(model_name, version)
        else:
            model_registry.promote_to_production(model_name, version)
        
        return {"status": "success", "model": model_name, "version": version, "stage": stage}
    
    return app


# =============================================================================
# SECTION 6: Model Monitoring and Drift Detection
# =============================================================================

"""
Model monitoring is essential for maintaining performance in production.
This section implements comprehensive monitoring including:

1. Data drift detection (feature distribution changes)
2. Concept drift detection (prediction distribution changes)
3. Performance drift detection (accuracy changes)
4. Alerting based on drift thresholds
5. Automated retraining triggers
"""

class ModelMonitor:
    """
    Monitor model performance and detect drift in production.
    
    This class provides comprehensive monitoring capabilities:
    - Data drift detection using statistical tests
    - Concept drift detection using prediction distributions
    - Performance monitoring against baseline
    - Alert generation for anomalies
    - Automated retraining recommendations
    
    Attributes:
        reference_data: Baseline data for comparison
        reference_predictions: Baseline predictions
        drift_thresholds: Thresholds for drift detection
        alert_history: History of generated alerts
        monitoring_window: Window size for monitoring
    """
    
    def __init__(
        self,
        reference_data: pd.DataFrame,
        reference_predictions: np.ndarray,
        drift_thresholds: Dict[str, float] = None,
        monitoring_window: int = 1000
    ):
        """
        Initialize model monitor.
        
        Args:
            reference_data: Baseline data for drift comparison
            reference_predictions: Baseline predictions
            drift_thresholds: Thresholds for drift detection
            monitoring_window: Window size for monitoring
        """
        self.reference_data = reference_data
        self.reference_predictions = reference_predictions
        self.monitoring_window = monitoring_window
        
        # Default drift thresholds
        self.drift_thresholds = drift_thresholds or {
            'ks_statistic': 0.1,  # Kolmogorov-Smirnov test
            'psi': 0.2,  # Population Stability Index
            'chi_square': 0.05,  # Chi-square p-value
            'accuracy_drop': 0.05,  # 5% accuracy drop
            'feature_drift': 0.1  # Feature drift threshold
        }
        
        self.alert_history = []
        self.drift_scores = []
        self.performance_history = []
        
        self.logger = logging.getLogger(f"{__name__}.ModelMonitor")
        
        # Calculate baseline statistics
        self.baseline_stats = self._calculate_baseline_stats()
    
    def _calculate_baseline_stats(self) -> Dict:
        """
        Calculate baseline statistics for reference data.
        
        Returns:
            Dictionary of baseline statistics
        """
        stats = {}
        
        # Numerical feature statistics
        numerical_cols = self.reference_data.select_dtypes(include=[np.number]).columns
        for col in numerical_cols:
            stats[col] = {
                'mean': self.reference_data[col].mean(),
                'std': self.reference_data[col].std(),
                'quantiles': self.reference_data[col].quantile([0.25, 0.5, 0.75]).to_dict()
            }
        
        # Categorical feature statistics
        categorical_cols = self.reference_data.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            stats[col] = {
                'value_counts': self.reference_data[col].value_counts(normalize=True).to_dict()
            }
        
        # Prediction statistics
        stats['predictions'] = {
            'fraud_rate': self.reference_predictions.mean(),
            'distribution': pd.Series(self.reference_predictions).value_counts(normalize=True).to_dict()
        }
        
        return stats
    
    def detect_data_drift(self, current_data: pd.DataFrame) -> Dict:
        """
        Detect drift in feature distributions.
        
        This method uses multiple statistical tests to detect
        changes in feature distributions:
        - Kolmogorov-Smirnov test for numerical features
        - Chi-square test for categorical features
        - Population Stability Index (PSI) for overall stability
        
        Args:
            current_data: Current batch of data
            
        Returns:
            Dictionary with drift detection results
        """
        drift_results = {
            'drift_detected': False,
            'feature_drifts': {},
            'overall_psi': 0,
            'drifted_features': []
        }
        
        # Check numerical features
        numerical_cols = current_data.select_dtypes(include=[np.number]).columns
        for col in numerical_cols:
            if col in self.reference_data.columns:
                # Kolmogorov-Smirnov test
                ks_statistic, ks_pvalue = ks_2samp(
                    self.reference_data[col].dropna(),
                    current_data[col].dropna()
                )
                
                # Calculate PSI for this feature
                psi = self._calculate_psi(
                    self.reference_data[col].dropna(),
                    current_data[col].dropna()
                )
                
                feature_drift = {
                    'ks_statistic': ks_statistic,
                    'ks_pvalue': ks_pvalue,
                    'psi': psi,
                    'drift_detected': (
                        ks_statistic > self.drift_thresholds['ks_statistic'] or
                        psi > self.drift_thresholds['psi']
                    )
                }
                
                drift_results['feature_drifts'][col] = feature_drift
                
                if feature_drift['drift_detected']:
                    drift_results['drifted_features'].append(col)
                    drift_results['drift_detected'] = True
        
        # Check categorical features
        categorical_cols = current_data.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if col in self.reference_data.columns:
                # Create contingency table
                ref_counts = self.reference_data[col].value_counts()
                curr_counts = current_data[col].value_counts()
                
                # Align categories
                all_categories = set(ref_counts.index) | set(curr_counts.index)
                ref_aligned = [ref_counts.get(cat, 0) for cat in all_categories]
                curr_aligned = [curr_counts.get(cat, 0) for cat in all_categories]
                
                # Chi-square test
                if len(all_categories) > 1:
                    chi2, pvalue, dof, expected = chi2_contingency([ref_aligned, curr_aligned])
                    
                    feature_drift = {
                        'chi2_statistic': chi2,
                        'chi2_pvalue': pvalue,
                        'drift_detected': pvalue < self.drift_thresholds['chi_square']
                    }
                    
                    drift_results['feature_drifts'][col] = feature_drift
                    
                    if feature_drift['drift_detected']:
                        drift_results['drifted_features'].append(col)
                        drift_results['drift_detected'] = True
        
        # Calculate overall PSI
        drift_results['overall_psi'] = np.mean([
            v['psi'] for v in drift_results['feature_drifts'].values()
            if 'psi' in v
        ])
        
        # Store drift score
        self.drift_scores.append({
            'timestamp': datetime.datetime.now().isoformat(),
            'drift_detected': drift_results['drift_detected'],
            'overall_psi': drift_results['overall_psi'],
            'num_drifted_features': len(drift_results['drifted_features'])
        })
        
        # Generate alert if drift detected
        if drift_results['drift_detected']:
            self._generate_alert('data_drift', drift_results)
        
        return drift_results
    
    def detect_concept_drift(self, current_predictions: np.ndarray) -> Dict:
        """
        Detect drift in prediction distributions.
        
        Args:
            current_predictions: Current batch of predictions
            
        Returns:
            Dictionary with concept drift results
        """
        # Calculate current fraud rate
        current_fraud_rate = current_predictions.mean()
        baseline_fraud_rate = self.reference_predictions.mean()
        
        # Calculate PSI for predictions
        psi = self._calculate_psi(
            pd.Series(self.reference_predictions),
            pd.Series(current_predictions)
        )
        
        # Detect drift
        fraud_rate_change = abs(current_fraud_rate - baseline_fraud_rate)
        drift_detected = (
            fraud_rate_change > self.drift_thresholds['accuracy_drop'] or
            psi > self.drift_thresholds['psi']
        )
        
        results = {
            'drift_detected': drift_detected,
            'baseline_fraud_rate': baseline_fraud_rate,
            'current_fraud_rate': current_fraud_rate,
            'fraud_rate_change': fraud_rate_change,
            'psi': psi
        }
        
        if drift_detected:
            self._generate_alert('concept_drift', results)
        
        return results
    
    def detect_performance_drift(
        self,
        current_data: pd.DataFrame,
        current_labels: np.ndarray,
        model_wrapper: ProductionModelWrapper
    ) -> Dict:
        """
        Detect drift in model performance.
        
        Args:
            current_data: Current batch of data
            current_labels: True labels
            model_wrapper: Production model wrapper
            
        Returns:
            Dictionary with performance drift results
        """
        # Get predictions
        predictions = model_wrapper.predict(current_data)
        probabilities = model_wrapper.predict_proba(current_data)
        
        # Calculate current metrics
        current_metrics = {
            'accuracy': accuracy_score(current_labels, predictions),
            'precision': precision_score(current_labels, predictions, zero_division=0),
            'recall': recall_score(current_labels, predictions, zero_division=0),
            'f1': f1_score(current_labels, predictions, zero_division=0),
            'roc_auc': roc_auc_score(current_labels, probabilities)
        }
        
        # Get baseline metrics from model wrapper
        baseline_metrics = model_wrapper.metrics
        
        # Detect performance drops
        performance_drift = {}
        drift_detected = False
        
        for metric, current_value in current_metrics.items():
            baseline_key = f'validation_{metric}' if f'validation_{metric}' in baseline_metrics else metric
            baseline_value = baseline_metrics.get(baseline_key, 0)
            
            if baseline_value > 0:
                relative_change = abs(current_value - baseline_value) / baseline_value
                
                performance_drift[metric] = {
                    'baseline': baseline_value,
                    'current': current_value,
                    'change': relative_change,
                    'drift_detected': relative_change > self.drift_thresholds['accuracy_drop']
                }
                
                if relative_change > self.drift_thresholds['accuracy_drop']:
                    drift_detected = True
        
        results = {
            'drift_detected': drift_detected,
            'performance_drift': performance_drift,
            'timestamp': datetime.datetime.now().isoformat()
        }
        
        # Store performance history
        self.performance_history.append({
            'timestamp': datetime.datetime.now().isoformat(),
            'metrics': current_metrics
        })
        
        if drift_detected:
            self._generate_alert('performance_drift', results)
        
        return results
    
    def _calculate_psi(self, expected: pd.Series, actual: pd.Series, bins: int = 10) -> float:
        """
        Calculate Population Stability Index (PSI).
        
        PSI measures the stability of a distribution over time.
        PSI < 0.1: No significant change
        0.1 <= PSI < 0.2: Moderate change
        PSI >= 0.2: Significant change
        
        Args:
            expected: Expected distribution (baseline)
            actual: Actual distribution (current)
            bins: Number of bins for continuous variables
            
        Returns:
            PSI value
        """
        # Discretize if continuous
        if expected.dtype.kind in 'fc':
            # Create bins based on expected distribution
            percentiles = np.linspace(0, 100, bins + 1)
            bin_edges = np.percentile(expected.dropna(), percentiles)
            bin_edges[0] = -np.inf
            bin_edges[-1] = np.inf
            
            expected_binned = pd.cut(expected, bins=bin_edges, labels=False)
            actual_binned = pd.cut(actual, bins=bin_edges, labels=False)
            
            expected_counts = expected_binned.value_counts(normalize=True).sort_index()
            actual_counts = actual_binned.value_counts(normalize=True).sort_index()
        else:
            # Categorical - use categories
            expected_counts = expected.value_counts(normalize=True)
            actual_counts = actual.value_counts(normalize=True)
        
        # Align indices
        all_categories = set(expected_counts.index) | set(actual_counts.index)
        expected_aligned = [expected_counts.get(cat, 0) for cat in all_categories]
        actual_aligned = [actual_counts.get(cat, 0) for cat in all_categories]
        
        # Add small epsilon to avoid division by zero
        expected_aligned = np.array(expected_aligned) + 1e-6
        actual_aligned = np.array(actual_aligned) + 1e-6
        
        # Normalize
        expected_aligned = expected_aligned / expected_aligned.sum()
        actual_aligned = actual_aligned / actual_aligned.sum()
        
        # Calculate PSI
        psi = np.sum((actual_aligned - expected_aligned) * np.log(actual_aligned / expected_aligned))
        
        return psi
    
    def _generate_alert(self, alert_type: str, details: Dict):
        """
        Generate an alert for drift detection.
        
        Args:
            alert_type: Type of alert (data_drift/concept_drift/performance_drift)
            details: Alert details
        """
        alert = {
            'alert_id': hashlib.md5(
                f"{alert_type}_{datetime.datetime.now().isoformat()}".encode()
            ).hexdigest()[:8],
            'type': alert_type,
            'timestamp': datetime.datetime.now().isoformat(),
            'details': details,
            'severity': self._calculate_severity(details)
        }
        
        self.alert_history.append(alert)
        
        # Log alert
        log_message = f"ALERT [{alert['severity']}] {alert_type} detected: {details}"
        if alert['severity'] == 'HIGH':
            self.logger.error(log_message)
        elif alert['severity'] == 'MEDIUM':
            self.logger.warning(log_message)
        else:
            self.logger.info(log_message)
        
        # Here you could integrate with alerting systems like:
        # - Slack webhooks
        # - Email notifications
        # - PagerDuty
        # - Prometheus Alertmanager
    
    def _calculate_severity(self, details: Dict) -> str:
        """
        Calculate alert severity based on drift magnitude.
        
        Args:
            details: Alert details
            
        Returns:
            Severity level (LOW/MEDIUM/HIGH)
        """
        if 'overall_psi' in details:
            if details['overall_psi'] > 0.3:
                return 'HIGH'
            elif details['overall_psi'] > 0.2:
                return 'MEDIUM'
        
        if 'fraud_rate_change' in details:
            if details['fraud_rate_change'] > 0.2:
                return 'HIGH'
            elif details['fraud_rate_change'] > 0.1:
                return 'MEDIUM'
        
        if 'num_drifted_features' in details:
            if details['num_drifted_features'] > 5:
                return 'HIGH'
            elif details['num_drifted_features'] > 2:
                return 'MEDIUM'
        
        return 'LOW'
    
    def get_monitoring_report(self) -> Dict:
        """
        Generate comprehensive monitoring report.
        
        Returns:
            Dictionary with monitoring report
        """
        return {
            'summary': {
                'total_alerts': len(self.alert_history),
                'high_severity_alerts': len([a for a in self.alert_history if a['severity'] == 'HIGH']),
                'medium_severity_alerts': len([a for a in self.alert_history if a['severity'] == 'MEDIUM']),
                'low_severity_alerts': len([a for a in self.alert_history if a['severity'] == 'LOW']),
                'drift_scores': self.drift_scores[-10:] if self.drift_scores else [],
                'performance_history': self.performance_history[-10:] if self.performance_history else []
            },
            'alerts': self.alert_history[-20:],  # Last 20 alerts
            'baseline_stats': self.baseline_stats
        }


# =============================================================================
# SECTION 7: Load Testing with Locust
# =============================================================================

"""
Load testing is crucial for understanding system behavior under stress.
This section implements comprehensive load testing using Locust to:
1. Simulate real-world traffic patterns
2. Measure latency under load
3. Identify bottlenecks
4. Determine system capacity
5. Validate auto-scaling behavior
"""

# Create locustfile.py for load testing
locustfile_content = '''
# locustfile.py
from locust import HttpUser, task, between
import random
import json
import numpy as np

class FraudDetectionUser(HttpUser):
    """
    Simulated user for load testing the fraud detection API.
    """
    
    wait_time = between(0.1, 0.5)  # Wait between requests
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.transactions = self._generate_test_data(1000)
        self.transaction_index = 0
    
    def _generate_test_data(self, n_samples: int) -> list:
        """
        Generate realistic test transactions.
        
        Args:
            n_samples: Number of samples to generate
            
        Returns:
            List of transaction dictionaries
        """
        transactions = []
        
        # Merchant categories
        categories = ['retail', 'grocery', 'restaurant', 'travel', 'entertainment', 'utilities']
        
        # Countries
        countries = ['US', 'GB', 'CA', 'FR', 'DE', 'JP', 'AU']
        
        # Cities
        cities = ['New York', 'London', 'Toronto', 'Paris', 'Berlin', 'Tokyo', 'Sydney']
        
        # Device types
        devices = ['mobile', 'desktop', 'tablet', 'pos']
        
        for i in range(n_samples):
            # Generate realistic transaction
            transaction = {
                'transaction_id': f"TXN_{i:08d}",
                'customer_id': f"CUST_{random.randint(1, 10000):06d}",
                'amount': round(random.uniform(1, 5000), 2),
                'currency': random.choice(['USD', 'EUR', 'GBP']),
                'transaction_time': datetime.datetime.now().isoformat(),
                'merchant_id': f"MERCH_{random.randint(1, 1000):04d}",
                'merchant_category': random.choice(categories),
                'country': random.choice(countries),
                'city': random.choice(cities),
                'device_id': f"DEV_{random.randint(1, 50000):06d}",
                'device_type': random.choice(devices),
                'ip_address': f"{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}"
            }
            
            # Occasionally create high-risk transactions
            if random.random() < 0.05:  # 5% high-risk
                transaction['amount'] = round(random.uniform(5000, 50000), 2)
                transaction['country'] = 'XX'  # Unknown country
                transaction['device_type'] = 'unknown'
            
            transactions.append(transaction)
        
        return transactions
    
    @task(3)
    def predict_single(self):
        """Test single prediction endpoint"""
        # Get next transaction (cyclic)
        transaction = self.transactions[self.transaction_index % len(self.transactions)]
        self.transaction_index += 1
        
        # Prepare request
        request_data = {
            'transaction': transaction,
            'explain': random.random() < 0.1  # 10% requests ask for explanation
        }
        
        # Make request
        with self.client.post(
            "/api/v1/predict",
            json=request_data,
            catch_response=True
        ) as response:
            if response.status_code == 200:
                response.success()
            else:
                response.failure(f"Status code: {response.status_code}")
    
    @task(1)
    def predict_batch(self):
        """Test batch prediction endpoint"""
        # Get batch of transactions
        batch_size = random.randint(5, 20)
        start_idx = self.transaction_index % len(self.transactions)
        batch = []
        
        for i in range(batch_size):
            idx = (start_idx + i) % len(self.transactions)
            batch.append(self.transactions[idx])
        
        self.transaction_index += batch_size
        
        # Prepare request
        request_data = {
            'transactions': batch,
            'batch_size': 10
        }
        
        # Make request
        with self.client.post(
            "/api/v1/predict/batch",
            json=request_data,
            catch_response=True
        ) as response:
            if response.status_code == 200:
                response.success()
            else:
                response.failure(f"Status code: {response.status_code}")
    
    @task(1)
    def health_check(self):
        """Test health endpoint"""
        self.client.get("/health")
    
    def on_start(self):
        """Called when a user starts"""
        # Warm up - make initial request
        self.predict_single()
'''


# =============================================================================
# SECTION 8: Main Execution and Testing
# =============================================================================

"""
This section demonstrates the complete workflow from model training
to production deployment, including:
1. Loading and preparing data
2. Training the model
3. Evaluating performance
4. Saving the model
5. Creating the API
6. Running load tests
7. Setting up monitoring
"""

def main():
    """
    Main execution function demonstrating the complete workflow.
    """
    print("=" * 80)
    print("VeritasFinancial - Fraud Detection Production Preparation")
    print("=" * 80)
    
    # Step 1: Load configuration
    print("\n1. Loading configuration...")
    model_config = ModelConfig(
        model_name="fraud_detection_xgboost",
        model_version="1.0.0",
        model_type="xgboost",
        input_features=[],  # Will be set from data
        hyperparameters={
            "n_estimators": 100,
            "max_depth": 6,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "scale_pos_weight": "auto",
            "random_state": 42
        }
    )
    
    # Step 2: Load and prepare data
    print("\n2. Loading and preparing data...")
    # In production, this would load from actual data sources
    # For demonstration, we'll generate synthetic data
    np.random.seed(42)
    n_samples = 10000
    
    # Generate synthetic features
    X = pd.DataFrame({
        'amount': np.random.exponential(100, n_samples),
        'hour_of_day': np.random.randint(0, 24, n_samples),
        'day_of_week': np.random.randint(0, 7, n_samples),
        'is_weekend': np.random.randint(0, 2, n_samples),
        'distance_from_home': np.random.exponential(50, n_samples),
        'transaction_velocity': np.random.poisson(5, n_samples),
        'device_risk_score': np.random.beta(1, 5, n_samples),
        'merchant_risk_score': np.random.beta(2, 8, n_samples),
        'country_risk_score': np.random.beta(1, 10, n_samples),
        'customer_age_months': np.random.exponential(24, n_samples),
        'avg_transaction_amount': np.random.exponential(80, n_samples),
        'std_transaction_amount': np.random.exponential(30, n_samples)
    })
    
    # Generate labels (fraud ~ 2%)
    fraud_prob = 1 / (1 + np.exp(-(
        0.1 * (X['amount'] - 100) / 100 +
        0.5 * X['device_risk_score'] +
        0.3 * X['merchant_risk_score'] +
        0.2 * X['country_risk_score'] -
        0.1 * X['customer_age_months'] / 12
    )))
    y = (np.random.random(n_samples) < fraud_prob).astype(int)
    
    print(f"Data loaded: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"Fraud rate: {y.mean():.4f} ({y.sum()} fraud cases)")
    
    # Set feature lists
    model_config.input_features = X.columns.tolist()
    model_config.numerical_features = X.columns.tolist()
    
    # Step 3: Split data
    print("\n3. Splitting data...")
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )
    
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    # Step 4: Create and train model
    print("\n4. Training model...")
    model_wrapper = ProductionModelWrapper(model_config)
    
    # Fit preprocessor
    model_wrapper.fit_preprocessor(X_train, y_train)
    
    # Train model
    metrics = model_wrapper.train(X_train, y_train, X_val, y_val)
    print(f"Training metrics: {metrics}")
    
    # Step 5: Evaluate on test set
    print("\n5. Evaluating on test set...")
    test_metrics = model_wrapper._calculate_model_metrics(
        model_wrapper.model, X_test, y_test, 'test'
    )
    print(f"Test metrics: {test_metrics}")
    
    # Step 6: Save model
    print("\n6. Saving model...")
    model_path = "models/production/fraud_detection_model_v1.pkl"
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    model_wrapper.save(model_path)
    print(f"Model saved to {model_path}")
    
    # Step 7: Register model
    print("\n7. Registering model...")
    registry = ModelRegistry()
    model_id = registry.register_model(
        model_path=model_path,
        model_name=model_config.model_name,
        version=model_config.model_version,
        metrics=test_metrics,
        tags={'environment': 'development', 'team': 'data-science'}
    )
    print(f"Model registered with ID: {model_id}")
    
    # Step 8: Promote to staging
    print("\n8. Promoting to staging...")
    registry.promote_to_staging(model_config.model_name, model_config.model_version)
    print("Model promoted to staging")
    
    # Step 9: Create monitoring baseline
    print("\n9. Creating monitoring baseline...")
    # Get predictions on validation set
    val_predictions = model_wrapper.predict(X_val)
    
    monitor = ModelMonitor(
        reference_data=X_val,
        reference_predictions=val_predictions
    )
    print("Monitoring baseline created")
    
    # Step 10: Test drift detection
    print("\n10. Testing drift detection...")
    # Create slightly drifted data
    X_drifted = X_test.copy()
    X_drifted['amount'] = X_drifted['amount'] * 1.5  # 50% increase in amounts
    X_drifted['device_risk_score'] = X_drifted['device_risk_score'] * 0.8  # Lower device risk
    
    drift_results = monitor.detect_data_drift(X_drifted)
    print(f"Data drift detected: {drift_results['drift_detected']}")
    print(f"Drifted features: {drift_results['drifted_features']}")
    
    # Step 11: Test concept drift
    print("\n11. Testing concept drift...")
    drifted_predictions = model_wrapper.predict(X_drifted)
    concept_drift = monitor.detect_concept_drift(drifted_predictions)
    print(f"Concept drift detected: {concept_drift['drift_detected']}")
    print(f"Fraud rate change: {concept_drift['fraud_rate_change']:.4f}")
    
    # Step 12: Generate monitoring report
    print("\n12. Generating monitoring report...")
    report = monitor.get_monitoring_report()
    print(f"Total alerts: {report['summary']['total_alerts']}")
    
    # Step 13: Create FastAPI app
    print("\n13. Creating FastAPI application...")
    app = create_app(
        model_path=model_path,
        registry_path="models/registry",
        config=DeploymentConfig(environment="development")
    )
    print("FastAPI application created")
    
    # Step 14: Test API locally
    print("\n14. Testing API locally...")
    from fastapi.testclient import TestClient
    
    client = TestClient(app)
    
    # Test health endpoint
    response = client.get("/health")
    print(f"Health check: {response.json()}")
    
    # Test prediction endpoint
    test_transaction = {
        'transaction_id': 'TXN_TEST_001',
        'customer_id': 'CUST_TEST_001',
        'amount': 1500.00,
        'currency': 'USD',
        'transaction_time': datetime.datetime.now().isoformat(),
        'merchant_id': 'MERCH_TEST_001',
        'merchant_category': 'retail',
        'country': 'US',
        'city': 'New York',
        'device_id': 'DEV_TEST_001',
        'device_type': 'mobile',
        'ip_address': '192.168.1.1'
    }
    
    response = client.post(
        "/api/v1/predict",
        json={'transaction': test_transaction, 'explain': True}
    )
    print(f"Prediction result: {response.json()}")
    
    # Step 15: Generate deployment instructions
    print("\n15. Deployment Instructions:")
    print("=" * 40)
    print("To deploy the model as a production API:")
    print("\n  # Start the API server")
    print("  uvicorn src.deployment.api:app --host 0.0.0.0 --port 8000 --workers 4")
    print("\n  # Or using Docker")
    print("  docker build -t fraud-detection-api .")
    print("  docker run -p 8000:8000 fraud-detection-api")
    print("\n  # Run load tests")
    print("  locust -f locustfile.py --host=http://localhost:8000")
    print("\n  # Monitor metrics")
    print("  curl http://localhost:8000/metrics")
    
    print("\n" + "=" * 80)
    print("Production preparation completed successfully!")
    print("=" * 80)
    
    return {
        'model_wrapper': model_wrapper,
        'registry': registry,
        'monitor': monitor,
        'app': app,
        'metrics': test_metrics
    }


if __name__ == "__main__":
    # Create logs directory if it doesn't exist
    os.makedirs("logs", exist_ok=True)
    
    # Run main function
    results = main()
    
    print("\nNotebook execution completed. The model is now ready for production deployment!")